In [ ]:
%%capture
!pip install yfinance pandas scikit-learn tensorflow requests

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import pickle
from datetime import datetime, timedelta
import time
import requests
import io
import os

print("Libraries loaded!")

In [ ]:
BACKEND_URL = "http://localhost:8000"  
TRAINING_INTERVAL_HOURS = 24  
NUM_MODELS = 3  
DAYS_OF_DATA = 365  

print(f"Backend URL: {BACKEND_URL}")
print(f"Training interval: {TRAINING_INTERVAL_HOURS} hours")
print(f"Number of models: {NUM_MODELS}")

In [ ]:
def create_sequences(data, window_size=60):
    xs, ys = [], []
    for i in range(len(data) - window_size):
        xs.append(data[i:(i + window_size)])
        ys.append(data[i + window_size][0])  # Predict Close price
    return np.array(xs), np.array(ys)

def build_model(input_shape, units=64):
    model = Sequential([
        LSTM(units, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(units//2, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
    return model

def fetch_data(ticker, days):
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    print(f"  Fetching {ticker} data from {start_date.date()} to {end_date.date()}...")
    df = yf.download(ticker, start=start_date, end=end_date, progress=False)
    if df.empty:
        raise RuntimeError("Failed to fetch data")
    print(f"  Downloaded {len(df)} rows")
    return df

In [ ]:
def train_single_model(df, model_id, units=64):
    print(f"\n[Model {model_id}] Training with {units} LSTM units...")
    features = df[['Close', 'Volume']].values.astype(np.float32)
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled = scaler.fit_transform(features)
    X, y = create_sequences(scaled, 60)
    print(f"  Sequences shape: {X.shape}")
    model = build_model((X.shape[1], X.shape[2]), units)